# Commit message classification with class imbalance

Reproduces the comparison from the imbalanced-learn example `examples/applications/plot_commit_message_classification.py`: a scikit-learn `Pipeline` (TF-IDF + `LinearSVC`) against an imbalanced-learn `Pipeline` with a `RandomOverSampler`, compared on per-class recall.

The data generation, labeling and cleaning functions are loaded from the example so both use the same code.

In [ ]:
import contextlib
import io
import runpy
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

EXAMPLE = Path("../imbalanced-learn/examples/applications/"
               "plot_commit_message_classification.py")

# Run the example quietly and keep its namespace.
with contextlib.redirect_stdout(io.StringIO()):
    ex = runpy.run_path(str(EXAMPLE))
plt.close("all")

RANDOM_STATE = ex["RANDOM_STATE"]
messages = ex["make_commit_messages"](
    1_000, {"fix": 0.7, "feature": 0.2, "other": 0.1}, random_state=RANDOM_STATE
)
y = np.array([ex["label_commit"](m) for m in messages])
X = np.array([ex["clean_message"](m) for m in messages])
print(Counter(y.tolist()))

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, random_state=RANDOM_STATE
)

## Baseline and resampled pipelines

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline

baseline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("svc", LinearSVC(random_state=RANDOM_STATE)),
])
resampled = ImbPipeline([
    ("tfidf", TfidfVectorizer()),
    ("oversample", RandomOverSampler(random_state=RANDOM_STATE)),
    ("svc", LinearSVC(random_state=RANDOM_STATE)),
])

predictions = {}
for name, model in [("scikit-learn Pipeline", baseline),
                    ("imblearn Pipeline + RandomOverSampler", resampled)]:
    predictions[name] = model.fit(X_train, y_train).predict(X_test)
    print(name)
    print(classification_report(y_test, predictions[name], zero_division=0))

## Per-class recall

In [ ]:
%matplotlib inline
from sklearn.metrics import accuracy_score, recall_score

labels = ["fix", "feature", "other"]
support = Counter(y_test.tolist())
x = np.arange(len(labels))
width = 0.38
fig, ax = plt.subplots(figsize=(7, 4.5))
for offset, (name, y_pred) in zip([-width / 2, width / 2], predictions.items()):
    recall = recall_score(y_test, y_pred, labels=labels, average=None)
    accuracy = accuracy_score(y_test, y_pred)
    bars = ax.bar(x + offset, recall, width, label=f"{name} (accuracy {accuracy:.2f})")
    ax.bar_label(bars, fmt="%.2f", padding=2)
ax.set_xticks(x, [f"{label}\n({support[label]} test commits)" for label in labels])
ax.set_ylim(0, 1.4)
ax.set_yticks(np.linspace(0, 1, 6))
ax.set_ylabel("Recall")
ax.set_title("Over-sampling trades recall on fixes for recall on minority classes")
ax.legend(loc="upper center")
fig.tight_layout()
plt.show()

Over-sampling raises the recall of "feature" and "other" at the cost of the recall of "fix", the precision of the minority classes, and overall accuracy. See the limitations at the end of the example: the data are synthetic and the template vocabulary makes the task easier than real commit histories.